In [0]:
# Importing and creating Spark Sesssion
from pyspark.sql import  SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = SparkSession.builder \
    .appName("Production_ETL") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

In [0]:
# Loading the delta tables 
instagram_df = spark.read.table("instagram.bronzelayer.instagram_usage_lifestyle")
print(instagram_df.printSchema())
# instagram_df.show()
print("Columns in Instagram DF : ", instagram_df.columns)

null_counts=instagram_df.select([
    F.count(F.when(F.col(column).isNull(),column)).alias(column)
    for column in instagram_df.columns
])
null_counts.show()

In [0]:
# function to replace null with zero
def replace_null_with_zero(df,columns):
    for column in columns:
        df = df.withColumn(
            column, 
            F.coalesce(F.col(column), F.lit(0))
            )
    return df

# function to replace negative
def clean_numeric_data(df,columns):
    for column in columns:
        df=df.withColumn(
            column,
             F.abs(F.coalesce(F.col(column),F.lit(0)))
             )
    return df

numeric_types=['int','bigint','double','float','decimal']
numeric_columns=[
    column for column,dtype in instagram_df.dtypes 
    if any(t in dtype for t  in numeric_types)
]
string_types=["string"]
string_columns=[
    column for column,dtype in instagram_df.dtypes 
    if  dtype  =="string"
]
instagram_df =instagram_df.dropna(subset=string_columns)
# instagram_df = replace_null_with_zero(instagram_df,numeric_columns)
instagram_df = clean_numeric_data(instagram_df,numeric_columns)

In [0]:
null_counts=instagram_df.select([
    F.count(F.when(F.col(column).isNull(),column)).alias(column)
    for column in instagram_df.columns
])
null_counts.show()
instagram_df.head(5)

In [0]:
is_unmarried_with_kids = (F.lower(F.col("relationship_status")) == "unmarried") & (F.col("has_children") == "Yes")
is_underage_married = (F.lower(F.col("relationship_status")) == "married") & (F.col("age") < 18)

is_invalid_age = (F.col("age") < 18) | (F.col("age") >= 100)
is_missing_id = F.col("user_id").isNull()
is_invalid_income = F.col("income_level").isNull()

# Create the flagging column
instagram_df = instagram_df.withColumn("rejection_reason", 
    F.when(is_missing_id, "Missing User ID")
     .when(is_invalid_age, "Age Out of Bounds")
     .when(is_unmarried_with_kids, "Logic Conflict: Unmarried with Kids")
     .when(is_underage_married, "Logic Conflict: Underage Married")
     .when(is_invalid_income, "Invalid Income Level")
     .otherwise("Valid")
)

# Add a simple boolean flag for quick filtering
instagram_df = instagram_df.withColumn("is_valid", F.col("rejection_reason") == "Valid")
rejected_df = instagram_df.withColumn("is_valid", F.col("rejection_reason") != "Valid")
